In [1]:
import matplotlib.pyplot as plt
import mygene
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# TCGA Preprocessing Summary

- **Load data**: Import mRNA expression (TPM) and clinical files, merge by sample ID.  
- **Filter**: Remove non-drug treatments (e.g., vaccines, “Other”) and toxicity-stopped cases.  
- **Labeling**: Convert treatment outcomes into RECIST labels (CR/PR=1, SD/PD=0, else=Unknown).  
- **Deduplicate**: Keep unique (project, case, diagnosis, treatment) entries.  
- **Expression processing**:  
  - Apply log2(TPM+1)  
  - Remove all-zero genes  
  - Select top 5000 most variable genes  
  - Gene-wise z-score normalization (mean 0, std 1)  
  - Cast to float32  
- **Save outputs**:  
  - `gene_exp.csv.gz` (expression matrix)  
  - `patients_info.csv.gz` (clinical + label info)  
- **Summary/visualization**: Count drug frequencies and plot drug × project heatmap.  


In [2]:
important_cols = [
    "project.project_id",
    "cases.submitter_id",
    "diagnoses.primary_diagnosis",
    "treatments.treatment_outcome",
    "treatments.therapeutic_agents",
]

In [3]:
non_smiles_compounds = [
    "Not Reported",
    "Other",
    "Clinical Trial",
    "Clinical Trial Agent",
    "Hormone Therapy",
    "Total Androgen Blockade",
    "Dendritic Cell Vaccine",
    "PEP-3-KLH Conjugate Vaccine",
    "Antineoplastic Vaccine",
    "Multi-glioblastoma-peptide-targeting Autologous Dendritic Cell Vaccine ICT-107",
    "MAGE-A3 Peptide Vaccine",
    "AE37 Peptide/GM-CSF Vaccine",
    "Recombinant PRAME Protein Plus AS15 Adjuvant GSK2302025A",
    "Recombinant MAGE-3.1 Antigen",
    "Innate Immunostimulator rBBX-01",
    "Recombinant Adenovirus-p53 SCH-58500",
    "Light-Emitting Oncolytic Vaccinia Virus GL-ONC1",
]

In [4]:
project_map = {
    "acc": "acc_tcga_gdc",
    "blca": "blca_tcga_gdc",
    "brca": "brca_tcga_gdc",
    "cesc": "cesc_tcga_gdc",
    "chol": "chol_tcga_gdc",
    "coad": "coad_tcga_gdc",
    "dlbc": "dlbclnos_tcga_gdc",
    "esca": "esca_tcga_gdc",
    "gbm": "gbm_tcga_gdc",
    "hnsc": "hnsc_tcga_gdc",
    "kich": "chrcc_tcga_gdc",
    "kirc": "ccrcc_tcga_gdc",
    "kirp": "prcc_tcga_gdc",
    "lihc": "hcc_tcga_gdc",
    "luad": "luad_tcga_gdc",
    "lusc": "lusc_tcga_gdc",
    "ov": "hgsoc_tcga_gdc",
    "paad": "paad_tcga_gdc",
    "prad": "prad_tcga_gdc",
    "read": "read_tcga_gdc",
    "sarc": "soft_tissue_tcga_gdc",
    "skcm": "skcm_tcga_gdc",
    "stad": "stad_tcga_gdc",
    "tgct": "nsgct_tcga_gdc",
    "thca": "thpa_tcga_gdc",
    "thym": "thym_tcga_gdc",
    "ucec": "ucec_tcga_gdc",
    "ucs": "ucs_tcga_gdc",
    "uvm": "um_tcga_gdc",
    "meso": "plmeso_tcga_gdc",
    "laml": "aml_tcga_gdc",
    "lgg": "difg_tcga_gdc",
    # 'pcpg': # cBioPortal has difficulty using this, so skip it.
}

In [5]:
ls

__pycache__/            models/                 VAE_GDSC.ipynb
_data_statistics.ipynb  preprocess_GDSC.ipynb   VAE_GDSC.py
_stat_TCGA.ipynb        preprocess_TCGA.ipynb   VAE_TCGA.ipynb
dataset/                TCGA_all_genes.json     VAE_TCGA.py
GDSC_all_genes.json     TCGA_top_genes.json
GDSC_genes.json         Untitled.ipynb


In [6]:
all_dfs = []

for gdc_name, cbio_name in project_map.items():
    print(f"Processing {gdc_name} / {cbio_name}...")

    # mRNAデータ
    try:
        tmp = (
            pd.read_csv(
                f"dataset/tcga/gene_exp/{cbio_name}/data_mrna_seq_tpm.txt",
                sep="\t",
                index_col=0,
            )
            .T.rename(lambda x: x[:-4])
            .pipe(lambda df: df[~df.index.duplicated(keep=False)])
        )
        tmp = tmp.reset_index().rename(columns={"index": "cases.submitter_id"})
        print(f"  mRNA samples after preprocessing: {tmp.shape[0]}")
    except FileNotFoundError:
        print(f"  mRNA data not found for {cbio_name}, skipping...")
        continue

    # 臨床データ
    try:
        recist = pd.read_table(
            f"dataset/tcga/clinical/clinical.project-tcga-{gdc_name}.2025-07-03/clinical.tsv"
        )[important_cols]
        recist = recist[recist["treatments.therapeutic_agents"] != "'--"]
        print(f"  Clinical samples after filtering: {recist.shape[0]}")
    except FileNotFoundError:
        print(f"  Clinical data not found for {gdc_name}, skipping...")
        continue

    # マージ & プロジェクト名列を追加
    merged = recist.merge(tmp, on="cases.submitter_id")
    print(f"  Samples after merge: {merged.shape[0]}\n")
    all_dfs.append(merged)

# 全プロジェクトをまとめる
all_merged = pd.concat(all_dfs, axis=0, ignore_index=True)
all_merged = all_merged[
    ~all_merged["treatments.therapeutic_agents"].isin(non_smiles_compounds)
]
all_merged = all_merged[
    all_merged["treatments.treatment_outcome"] != "Treatment Stopped Due to Toxicity"
]

print("Final merged shape:", all_merged.shape)

Processing acc / acc_tcga_gdc...
  mRNA samples after preprocessing: 79
  Clinical samples after filtering: 152
  Samples after merge: 136

Processing blca / blca_tcga_gdc...
  mRNA samples after preprocessing: 406
  Clinical samples after filtering: 366
  Samples after merge: 359

Processing brca / brca_tcga_gdc...
  mRNA samples after preprocessing: 1083
  Clinical samples after filtering: 2448


/var/folders/y3/ssnk1ytd3m5bjmrchh2lt74srg76p8/T/ipykernel_75908/2676598862.py:25: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  recist = pd.read_table(


  Samples after merge: 2419

Processing cesc / cesc_tcga_gdc...
  mRNA samples after preprocessing: 302
  Clinical samples after filtering: 302
  Samples after merge: 294

Processing chol / chol_tcga_gdc...
  mRNA samples after preprocessing: 35
  Clinical samples after filtering: 14
  Samples after merge: 14

Processing coad / coad_tcga_gdc...
  mRNA samples after preprocessing: 456
  Clinical samples after filtering: 638
  Samples after merge: 624

Processing dlbc / dlbclnos_tcga_gdc...
  mRNA samples after preprocessing: 48
  Clinical samples after filtering: 262
  Samples after merge: 262

Processing esca / esca_tcga_gdc...
  mRNA samples after preprocessing: 183
  Clinical samples after filtering: 88
  Samples after merge: 86

Processing gbm / gbm_tcga_gdc...
  mRNA samples after preprocessing: 277
  Clinical samples after filtering: 1475
  Samples after merge: 785

Processing hnsc / hnsc_tcga_gdc...
  mRNA samples after preprocessing: 518
  Clinical samples after filtering: 353
 

In [7]:
def recist_to_label(x):
    if x in ["Complete Response", "Partial Response"]:
        return 1
    elif x in ["Stable Disease", "Progressive Disease", "No Response"]:
        return 0
    else:  # '--', 'Treatment Ongoing', 'Unknown', 'Not Reported'
        return "Unknown"


all_merged["recist_label"] = all_merged["treatments.treatment_outcome"].apply(
    recist_to_label
)

In [8]:
all_merged = all_merged.drop_duplicates(
    subset=[
        "project.project_id",
        "cases.submitter_id",
        "diagnoses.primary_diagnosis",
        "treatments.therapeutic_agents",
    ]
).reset_index(drop=True)
all_merged

,project.project_id,cases.submitter_id,diagnoses.primary_diagnosis,treatments.treatment_outcome,treatments.therapeutic_agents,1,10,100,1000,10000,...,9989,999,9990,9991,9992,9993,9994,9995,9997,recist_label
0,TCGA-ACC,TCGA-OR-A5LL,Adrenal cortical carcinoma,'--,Mitotane,0.0000,0.0773,11.4862,103.4051,23.7715,...,19.3521,0.0163,11.7529,13.8138,0.0936,61.7972,8.6883,0.0000,0.1438,Unknown
1,TCGA-ACC,TCGA-OR-A5J7,Adrenal cortical carcinoma,Progressive Disease,Mitotane,0.0418,0.1732,6.8554,0.2788,1.7781,...,14.8739,0.0091,6.6842,21.7122,0.1049,23.2751,5.0279,0.0000,0.0806,0
2,TCGA-ACC,TCGA-P6-A5OG,"Osteosarcoma, NOS",'--,Cisplatin,0.3781,0.3168,22.8133,0.8672,16.8202,...,35.8733,0.1715,6.5633,25.9265,0.0000,40.0989,4.5325,0.0000,1.2211,Unknown
3,TCGA-ACC,TCGA-P6-A5OG,"Osteosarcoma, NOS",'--,Methotrexate,0.3781,0.3168,22.8133,0.8672,16.8202,...,35.8733,0.1715,6.5633,25.9265,0.0000,40.0989,4.5325,0.0000,1.2211,Unknown
4,TCGA-ACC,TCGA-P6-A5OG,"Osteosarcoma, NOS",'--,Doxorubicin Hydrochloride,0.3781,0.3168,22.8133,0.8672,16.8202,...,35.8733,0.1715,6.5633,25.9265,0.0000,40.0989,4.5325,0.0000,1.2211,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10685,TCGA-LGG,TCGA-DU-6392,"Astrocytoma, anaplastic",'--,Procarbazine,0.5911,0.5088,4.2971,51.0387,20.9196,...,48.5935,0.9807,27.6984,18.2162,0.0000,228.9859,11.8467,0.2876,0.0728,Unknown
10686,TCGA-LGG,TCGA-DU-6392,"Astrocytoma, anaplastic",'--,Vincristine,0.5911,0.5088,4.2971,51.0387,20.9196,...,48.5935,0.9807,27.6984,18.2162,0.0000,228.9859,11.8467,0.2876,0.0728,Unknown
10687,TCGA-LGG,TCGA-HT-A74H,"Astrocytoma, anaplastic",Unknown,Temozolomide,0.2300,1.1367,6.8564,28.3516,9.8491,...,18.8761,2.1629,4.1599,8.7955,0.4079,81.2995,2.8023,0.0000,0.0392,Unknown
10688,TCGA-LGG,TCGA-HT-7470,"Oligodendroglioma, anaplastic",'--,Temozolomide,0.2252,3.9320,9.3855,54.7707,32.2815,...,30.1413,18.6072,21.2616,11.3950,0.0943,178.8629,10.1870,0.0409,0.0362,Unknown


In [9]:
len(all_merged['cases.submitter_id'].unique())

4416

In [10]:
len(all_merged['treatments.therapeutic_agents'].unique())

272

In [55]:
all_merged['treatments.therapeutic_agents'].unique()

array(['Mitotane', 'Cisplatin', 'Methotrexate',
       'Doxorubicin Hydrochloride', 'Carboplatin', 'Capecitabine',
       'Etoposide', 'Sorafenib', 'Doxorubicin', 'Bevacizumab',
       'Streptozocin', 'Sunitinib', 'Ketoconazole', 'Gemcitabine',
       'BCG Solution', 'Gemcitabine Hydrochloride', 'Paclitaxel',
       'Oxaliplatin', 'Docetaxel', 'Fluorouracil', 'Vinblastine',
       'Ifosfamide', 'Fosaprepitant', 'Unknown', 'Vandetanib',
       'Halichondrin B', 'Mesna', 'Vorinostat', 'Erlotinib Hydrochloride',
       'Temsirolimus', 'Vinorelbine', 'Nivolumab', 'Ipilimumab',
       'Anastrozole', 'Zoledronic Acid', 'Cyclophosphamide',
       'Trastuzumab', 'Exemestane', 'Tamoxifen', 'Nab-paclitaxel',
       'Letrozole', 'Epirubicin', 'Leuprolide Acetate', 'Prednisone',
       'Rituximab', 'Vincristine', 'Pegfilgrastim', 'Toremifene Citrate',
       'Goserelin', 'Leuprolide', 'Fulvestrant', 'Everolimus',
       'Triptorelin', 'Clodronate Disodium', 'Lapatinib', 'Metformin',
       'Gosere

In [42]:
name2smiles = pd.read_csv('../Aug8_nsc_cid_smiles.csv', usecols=['SMILES', 'NAME']).dropna()


0                                       p-Toluquinone
1                          4-Amino-3-pentadecylphenol
2        3-(Dimethylamino)propiophenone hydrochloride
3                                       Cycloheximide
4                                4-Phenylbutyric acid
                             ...                     
23881                                  piperlongumine
23892                                           FH535
23946                                         GW-2580
23955                                      GSK269962A
23979                                        YK 4-279
Name: NAME, Length: 23680, dtype: object

In [44]:
len(set(all_merged['treatments.therapeutic_agents'].unique()) & set(name2smiles['NAME']))

79

In [53]:
len(set([i.capitalize() for i in all_merged['treatments.therapeutic_agents'].unique()]) & set(name2smiles['NAME'].str.capitalize()))

95

{'1-amino-2-[4-(3-chloro-2-cyanophenoxy)phenyl]sulfonyl-3-(4-chlorophenyl)guanidine',
 "N'-(4-chlorophenyl)-2-aminobenzamidine",
 '4-[[[2-[(1s)-1-[(2-amino-5-cyano-6-methylpyrimidin-4-yl)amino]ethyl]-4-oxo-3-phenylquinazolin-5-yl]amino]methyl]-n-hydroxybenzamide',
 '6-bromo-2-methoxy-6,7,8,9-tetrahydro-5h-benzo[7]annulen-5-one',
 '(3,5-dichloro-2-hydroxy-benzylidene)-hydrazide',
 '11-nitronoracronycin',
 '1-[(4-methoxyphenyl)thio]-1-(1-methylethenyl)cyclopropane',
 '(e)-2-(benzenesulfonyl)-3-[2-(4-methoxyphenyl)-1h-indol-3-yl]prop-2-enenitrile',
 '4-[(e)-[[2-(4-methylphenyl)quinazolin-4-yl]hydrazinylidene]methyl]phenol',
 '2-((2-fluoro-4-iodophenyl)amino)-n-(2-hydroxyethoxy)-1,5-dimethyl-6-oxo-1,6-dihydropyridine-3-carboxamide',
 '2-(2-hydroxyethyl)-8-(4-methoxyphenyl)-5-oxa-2-azatetracyclo[7.7.0.03,7.011,15]hexadeca-1(9),3(7),10,15-tetraen-6-one',
 'Benzamide, n-(2,4-dimethylphenyl)-2-hydroxy-3-nitro-',
 '(5z)-5-[(4-hydroxy-3-methoxyphenyl)methylidene]-2-phenyl-3-(6-phenylsulfanyl-1h-

In [11]:
all_merged[
    [
        "project.project_id",
        "cases.submitter_id",
        "diagnoses.primary_diagnosis",
        "treatments.therapeutic_agents",
        "treatments.treatment_outcome",
        "recist_label",
    ]
].to_csv("dataset/tcga/patients_info.csv.gz", compression="gzip")

In [12]:
tmp = all_merged.drop(
    [
        "project.project_id",
        "diagnoses.primary_diagnosis",
        "treatments.therapeutic_agents",
        "treatments.treatment_outcome",
        "recist_label",
    ],
    axis=1,
).drop_duplicates()
tmp.head()

,cases.submitter_id,1,10,100,1000,10000,100008586,100009613,100009667,100009668,...,9988,9989,999,9990,9991,9992,9993,9994,9995,9997
0,TCGA-OR-A5LL,0.0000,0.0773,11.4862,103.4051,23.7715,0.0,0.0000,0.0000,0.0000,...,20.5672,19.3521,0.0163,11.7529,13.8138,0.0936,61.7972,8.6883,0.0000,0.1438
1,TCGA-OR-A5J7,0.0418,0.1732,6.8554,0.2788,1.7781,0.0,0.0554,0.0846,0.0562,...,8.9225,14.8739,0.0091,6.6842,21.7122,0.1049,23.2751,5.0279,0.0000,0.0806
2,TCGA-P6-A5OG,0.3781,0.3168,22.8133,0.8672,16.8202,0.0,0.1738,0.1768,0.2350,...,18.4651,35.8733,0.1715,6.5633,25.9265,0.0000,40.0989,4.5325,0.0000,1.2211
6,TCGA-OR-A5J8,0.0313,1.9463,26.6439,20.8490,10.7220,0.0,0.0623,0.0950,0.2526,...,11.6388,35.4112,0.0615,4.8078,26.5329,0.0000,90.1901,4.3745,0.0000,0.7244
7,TCGA-OR-A5JL,0.0430,0.2231,7.1247,0.6080,3.7718,0.0,0.0000,0.0000,0.0000,...,8.8566,9.5915,0.0188,9.8285,14.5799,0.6486,67.2278,4.0895,0.0469,0.2491


In [13]:
exp = tmp.drop('cases.submitter_id', axis=1)

print('max', np.max(exp.values))
print('min', np.min(exp.values))
print('median', np.median(exp.values))
print('mean', np.mean(exp.values))

max 798771.8008
min 0.0
median 0.225
mean 24.212680421727825


In [14]:
tmp = tmp.loc[:, tmp.sum() != 0]
entrez_ids = tmp.columns[1:].tolist()

mg = mygene.MyGeneInfo()
out = mg.querymany(entrez_ids, scopes="entrezgene", fields="symbol", species="human")

id_to_symbol = {int(x["query"]): x["symbol"] for x in out if "notfound" not in x}
valid_ids = list(id_to_symbol.keys())

# フィルタ & rename
tmp = tmp[["cases.submitter_id"] + valid_ids]
tmp = tmp.rename(columns=id_to_symbol)
tmp.head()

85 input query terms found no hit:	['100506377', '100507415', '100873943', '100874266', '100874283', '100874340', '100996306', '1009964


,cases.submitter_id,A1BG,NAT2,ADA,CDH2,AKT3,LINC02584,POU5F1P5,POU5F1P6,POU5F1P7,...,DMTF1,PPP4R1,CDH1,SLC12A6,PTBP3,KCNE2,DGCR2,CASP8AP2,ELK2BP,SCO2
0,TCGA-OR-A5LL,0.0000,0.0773,11.4862,103.4051,23.7715,0.0000,0.0000,0.0000,0.0,...,20.5672,19.3521,0.0163,11.7529,13.8138,0.0936,61.7972,8.6883,0.0000,0.1438
1,TCGA-OR-A5J7,0.0418,0.1732,6.8554,0.2788,1.7781,0.0554,0.0846,0.0562,0.0,...,8.9225,14.8739,0.0091,6.6842,21.7122,0.1049,23.2751,5.0279,0.0000,0.0806
2,TCGA-P6-A5OG,0.3781,0.3168,22.8133,0.8672,16.8202,0.1738,0.1768,0.2350,0.0,...,18.4651,35.8733,0.1715,6.5633,25.9265,0.0000,40.0989,4.5325,0.0000,1.2211
6,TCGA-OR-A5J8,0.0313,1.9463,26.6439,20.8490,10.7220,0.0623,0.0950,0.2526,0.0,...,11.6388,35.4112,0.0615,4.8078,26.5329,0.0000,90.1901,4.3745,0.0000,0.7244
7,TCGA-OR-A5JL,0.0430,0.2231,7.1247,0.6080,3.7718,0.0000,0.0000,0.0000,0.0,...,8.8566,9.5915,0.0188,9.8285,14.5799,0.6486,67.2278,4.0895,0.0469,0.2491


In [15]:
tmp.iloc[:, 1:] = np.log2(tmp.iloc[:, 1:] + 1)
top_genes = tmp.iloc[:, 1:].var().sort_values(ascending=False).head(5000).index
top_genes = sorted(top_genes)

In [17]:
print('max', np.max(tmp.iloc[:, 1:].values))
print('min', np.min(tmp.iloc[:, 1:].values))
print('median', np.median(tmp.iloc[:, 1:].values))
print('mean', np.mean(tmp.iloc[:, 1:].values))

max 19.607425682507834
min 0.0
median 0.34505567517929614
mean 1.6180300830954053


In [18]:
import json

with open("TCGA_top_genes.json", "w") as f:
    json.dump(top_genes, f)

In [19]:
with open("TCGA_all_genes.json", "w") as f:
    json.dump(list(tmp.columns[1:]), f)

In [20]:
with open("GDSC_genes.json", "r") as f:
    GDSC_top_genes = json.load(f)

with open("GDSC_all_genes.json", "r") as f:
    GDSC_all_genes = json.load(f)

In [21]:
all_genes = sorted(set(tmp.columns[1:]) & set(GDSC_all_genes))
top_genes = sorted(set(GDSC_top_genes) | set(top_genes))
genes = (sorted(set(top_genes) & set(all_genes)))

In [22]:
len(genes)

6254

In [23]:
tmp = pd.concat([tmp["cases.submitter_id"], tmp[genes]], axis=1)
eps = 1e-6
tmp.iloc[:, 1:] = (tmp.iloc[:, 1:] - tmp.iloc[:, 1:].mean()) / (tmp.iloc[:, 1:].std(ddof=0) + eps)
tmp.iloc[:, 1:] = tmp.iloc[:, 1:].astype(np.float32)

In [24]:
tmp.to_csv("dataset/tcga/gene_exp.csv.gz", compression="gzip")

,cases.submitter_id,A1BG,A1CF,A2M,A2ML1,A4GALT,AADAC,AADAT,AAMDC,AARD,...,ZNRF1,ZNRF3,ZP3,ZSCAN16,ZSCAN18,ZSCAN31,ZSWIM5,ZSWIM6,ZWINT,ZYX
0,TCGA-OR-A5LL,-0.738090,-0.358869,-1.284364,-0.662174,0.071062,3.991960,-1.507428,1.583543,1.452025,...,-0.035460,1.290557,-0.139355,-0.702613,-0.926826,0.604660,2.004359,-0.153295,-2.592233,-2.581240
1,TCGA-OR-A5J7,-0.579840,-0.383298,0.212061,-0.623405,-0.529421,-0.373525,-0.889171,4.461759,-0.721811,...,-0.683419,-0.812265,0.572096,-1.386443,-1.057612,-0.939410,0.140051,-0.955227,1.017701,-2.477873
2,TCGA-P6-A5OG,0.501265,-0.398129,0.612546,-0.707374,0.259545,-0.645832,0.036936,0.411848,-0.234270,...,0.116915,-1.172862,0.172996,-0.706473,0.292523,-2.336992,-1.764804,2.105065,1.142627,0.313310
6,TCGA-OR-A5J8,-0.618986,-0.398129,1.094104,-0.583765,1.155040,-0.674528,0.556962,0.336370,-0.568159,...,0.230676,-0.589812,-1.316565,-1.685371,0.263213,-1.948471,0.314336,-0.533600,0.018025,1.441008
7,TCGA-OR-A5JL,-0.575391,-0.309080,-0.257669,-0.000349,-0.863802,-0.056424,-0.560281,0.497837,-0.535786,...,-1.027670,0.259927,0.202159,-1.497235,-0.230091,0.249226,1.426305,-1.209734,-0.579039,-2.153087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10682,TCGA-DU-8168,-0.495089,-0.310761,-0.168205,0.669120,-1.439784,-0.648848,2.125469,0.043546,-0.577613,...,0.858557,0.673450,-2.524928,0.289409,1.074694,0.190407,0.922621,2.062846,0.496325,-0.730090
10683,TCGA-DU-6392,1.056665,-0.384697,1.255113,-0.552191,-0.663550,-0.674528,1.718347,-1.407837,0.141752,...,1.190066,0.320155,-0.236838,-0.011876,1.320254,-0.745764,1.267073,0.759521,0.014602,0.815270
10687,TCGA-HT-A74H,0.061909,-0.390948,0.128906,0.822252,-0.397182,-0.647815,0.600416,-0.256046,-0.570193,...,-1.134421,-0.352989,0.678500,-0.576134,1.564978,-0.707345,0.251465,-0.209969,-2.623774,-1.390789
10688,TCGA-HT-7470,0.046798,-0.391460,0.466210,0.916859,-1.036840,-0.674528,1.064147,-0.368759,-0.178688,...,0.047559,1.022071,-0.514832,-0.708323,1.325688,-0.075622,1.067387,1.337376,-1.864503,-1.692229
